# Medical Image Segmentation - Colab Training

Trains the U-Net from the `Medical Imaging` project on the SIIM-ACR pneumothorax subset (the same 400 images used for the local naive-baseline benchmark, so results are directly comparable), evaluates Dice/IoU against that baseline, and generates Seg-Grad-CAM explainability examples.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

**You will need:** your `kaggle.json` API token (Kaggle > Settings > API > Create New Token), and to have joined/accepted the rules for the `siim-acr-pneumothorax-segmentation` competition at kaggle.com (even though images come from a re-hosted dataset, the label CSV comes from the competition itself).

In [ ]:
!nvidia-smi

## Setup

In [ ]:
!pip install -q pydicom kaggle

In [ ]:
import os

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_path = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(kaggle_path):
    from google.colab import files
    print("Upload your kaggle.json (Kaggle > Settings > Create New Token):")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, kaggle_path)
os.chmod(kaggle_path, 0o600)
print("Kaggle credentials ready.")

## Recreate the `dicom_seg` package

These cells write out the exact same, already-tested modules from the local repo (`dicom_seg/`), via `%%writefile`, so training runs identical code to what `pytest` verified locally.

In [ ]:
!mkdir -p dicom_seg

In [ ]:
%%writefile dicom_seg/__init__.py


In [ ]:
%%writefile dicom_seg/preprocessing.py
"""
Milestone 1 & 2: DICOM loading, metadata inspection, intensity correction,
and CT windowing.

Pipeline for turning a raw DICOM file into a model-ready image:

    raw stored pixel values (arbitrary integers, NOT display-ready)
        -> Modality LUT (RescaleSlope/RescaleIntercept, or a LUT Sequence)
           gives real-world intensity units (e.g. Hounsfield Units for CT)
        -> VOI LUT / windowing (WindowCenter/WindowWidth, or a VOI LUT Sequence)
           maps the clinically relevant range onto full display contrast
        -> PhotometricInterpretation correction (MONOCHROME1 is inverted:
           the minimum stored value is meant to display as white)
        -> normalize to [0, 1] or [0, 255]

Getting the last two steps wrong is the classic bug: images render as
photographic negatives (MONOCHROME1 mistake) or with washed-out/blown-out
contrast (skipping rescale/windowing).
"""
from __future__ import annotations

from pathlib import Path
from typing import Union

import numpy as np
import pydicom
from pydicom import Dataset
from pydicom.pixels import apply_modality_lut, apply_voi_lut

PathLike = Union[str, Path]

# Fields worth inspecting on every DICOM before touching pixel data, with a
# one-line explanation of what each one means and why it matters.
METADATA_FIELDS = {
    "PatientAge": "Patient age at time of study (e.g. '034Y'). Demographic context only.",
    "Modality": "Acquisition device type: CT, CR/DX (X-ray), MR, US, etc. Determines "
                "what the stored pixel values physically represent.",
    "PhotometricInterpretation": "How stored pixel values map to displayed brightness. "
                "MONOCHROME2: low value = dark (normal). MONOCHROME1: low value = "
                "bright (inverted) -- get this wrong and images look like negatives.",
    "Rows": "Image height in pixels.",
    "Columns": "Image width in pixels.",
    "BitsStored": "Number of bits actually used per pixel sample (e.g. 12 bits packed "
                "into a 16-bit word). Values above 2**BitsStored-1 shouldn't occur.",
    "BitsAllocated": "Number of bits allocated per pixel sample in memory (usually 16).",
    "PixelRepresentation": "0 = unsigned integer pixel values, 1 = signed (two's "
                "complement). CT images are commonly signed to represent negative HU.",
    "RescaleSlope": "Modality LUT slope: real_value = stored_value * slope + intercept. "
                "Converts raw stored integers into physical units (e.g. Hounsfield Units).",
    "RescaleIntercept": "Modality LUT intercept (see RescaleSlope). For CT this is "
                "commonly -1024, since stored values are usually unsigned/offset "
                "while HU air is -1000.",
    "WindowCenter": "Center of the default display window (VOI LUT), in the same units "
                "as the rescaled pixel values.",
    "WindowWidth": "Width of the default display window. Narrower = higher contrast "
                "over a smaller intensity range.",
}


def load_dicom(path: PathLike) -> Dataset:
    """Read a DICOM file from disk."""
    return pydicom.dcmread(str(path))


def describe_metadata(ds: Dataset) -> dict:
    """Return the key metadata fields present on `ds`, with their values.

    Fields not present on this particular file are omitted (DICOM is a
    standard with many optional fields -- not every modality populates
    every tag).
    """
    return {
        field: getattr(ds, field)
        for field in METADATA_FIELDS
        if field in ds
    }


def get_modality_corrected(ds: Dataset) -> np.ndarray:
    """Raw stored pixel values -> real-world intensity units.

    Applies the Modality LUT: either a linear RescaleSlope/RescaleIntercept
    transform, or a Modality LUT Sequence, whichever the file provides. For
    CT this yields Hounsfield Units. For X-ray there usually is no rescale
    (slope=1, intercept=0), so this is close to a no-op but is still the
    correct step to apply before windowing.
    """
    raw = ds.pixel_array.astype(np.float64)
    return apply_modality_lut(raw, ds).astype(np.float64)


def apply_window(image: np.ndarray, center: float, width: float) -> np.ndarray:
    """Map the intensity window [center - width/2, center + width/2] to
    [0, 255], clipping everything outside it.

    This is the core windowing operation: CT stores a huge dynamic range
    (air ~ -1000 HU to dense bone ~ +1000-3000 HU) that no display can show
    with good contrast all at once. Picking a narrow window around the
    tissue of interest (e.g. center=-600, width=1500 for lung; center=400,
    width=1800 for bone) throws away irrelevant range in exchange for much
    higher contrast in the range that matters.
    """
    width = max(width, 1e-6)
    lower = center - width / 2.0
    upper = center + width / 2.0
    windowed = np.clip(image, lower, upper)
    windowed = (windowed - lower) / (upper - lower) * 255.0
    return windowed.astype(np.uint8)


def dicom_to_image(
    path_or_ds: Union[PathLike, Dataset],
    apply_voi: bool = True,
    output_range: str = "unit",
) -> np.ndarray:
    """Full Milestone-1 pipeline: DICOM -> correctly oriented, correctly
    scaled, normalized NumPy image.

    Steps:
      1. Read pixel data and apply the Modality LUT (rescale to real units).
      2. Apply the VOI LUT / default window, if present and requested
         (falls back to the raw modality-corrected values otherwise).
      3. Min-max normalize to [0, 1].
      4. Invert if PhotometricInterpretation is MONOCHROME1, so that the
         output always follows the "low value = dark" convention regardless
         of how the source file stored it.

    Parameters
    ----------
    path_or_ds: path to a .dcm file, or an already-loaded pydicom Dataset.
    apply_voi: whether to apply the file's VOI LUT/default window before
        normalizing. Set False to get a plain min-max normalization of the
        modality-corrected values instead (useful when you'll window
        yourself, e.g. for the lung/bone comparison in Milestone 2).
    output_range: "unit" for float32 in [0, 1], or "uint8" for [0, 255].
    """
    ds = path_or_ds if isinstance(path_or_ds, Dataset) else load_dicom(path_or_ds)

    modality_arr = get_modality_corrected(ds)
    arr = apply_voi_lut(modality_arr, ds) if apply_voi else modality_arr
    arr = arr.astype(np.float64)

    lo, hi = float(arr.min()), float(arr.max())
    if hi > lo:
        norm = (arr - lo) / (hi - lo)
    else:
        norm = np.zeros_like(arr)

    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        norm = 1.0 - norm

    if output_range == "uint8":
        return (norm * 255.0).astype(np.uint8)
    return norm.astype(np.float32)


In [ ]:
%%writefile dicom_seg/rle.py
"""
RLE (run-length encoding) mask decode/encode for the SIIM-ACR Pneumothorax
dataset.

Faithful re-implementation of Kaggle's official `mask_functions.py`
(`rle2mask`/`mask2rle`), which encodes runs while iterating pixels
column-by-column (outer loop over width, inner loop over height) and treats
each "start" value as a GAP length from the end of the previous run, not an
absolute pixel index (i.e. this is a delta-encoded RLE, not the
absolute-offset RLE used by e.g. COCO).

Because of that column-major traversal, decoding naively reshapes to
(width, height) -- the transpose to (height, width), matching a normal
image array's (rows, cols) axes, is applied here so callers get masks
already aligned with `dicom_to_image` output. Forgetting this transpose is
the single most common bug reported against this exact dataset.
"""
from __future__ import annotations

import numpy as np

EMPTY_MASK_TOKEN = "-1"


def rle_decode(rle_string: str, height: int, width: int) -> np.ndarray:
    """Decode one SIIM-style RLE string into a (height, width) uint8 mask
    of 0/1 values. Returns an all-zero mask for the "no pneumothorax"
    sentinel ("-1") or an empty string.
    """
    rle_string = rle_string.strip()
    if rle_string == "" or rle_string == EMPTY_MASK_TOKEN:
        return np.zeros((height, width), dtype=np.uint8)

    values = np.asarray([int(x) for x in rle_string.split()], dtype=np.int64)
    starts = values[0::2]
    lengths = values[1::2]

    flat = np.zeros(width * height, dtype=np.uint8)
    position = 0
    for start, length in zip(starts, lengths):
        position += int(start)
        flat[position:position + int(length)] = 1
        position += int(length)

    mask = flat.reshape(width, height)
    return mask.T


def rle_decode_multi(rle_strings: list[str], height: int, width: int) -> np.ndarray:
    """Decode and OR-combine multiple RLE strings for the same image (some
    images have several disconnected pneumothorax regions, one CSV row
    each).
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    for rle_string in rle_strings:
        mask |= rle_decode(rle_string, height, width)
    return mask


def rle_encode(mask: np.ndarray) -> str:
    """Encode a (height, width) 0/1 (or 0/255) mask back into a SIIM-style
    delta RLE string. Inverse of `rle_decode` for a single connected/simple
    mask; mainly used for round-trip testing.
    """
    width, height = mask.shape[1], mask.shape[0]
    binary = (mask.T.flatten() > 0).astype(np.uint8)  # column-major flatten via (width,height)

    if not binary.any():
        return EMPTY_MASK_TOKEN

    pairs = []
    last_end = 0  # position just after the previous run
    in_run = False
    run_start = 0
    for i, v in enumerate(binary):
        if v == 1 and not in_run:
            run_start = i
            in_run = True
        elif v == 0 and in_run:
            pairs.append((run_start - last_end, i - run_start))
            last_end = i
            in_run = False
    if in_run:
        pairs.append((run_start - last_end, len(binary) - run_start))

    return " ".join(f"{gap} {length}" for gap, length in pairs)


In [ ]:
%%writefile dicom_seg/dataset.py
"""
Milestone 3: segmentation dataset pipeline for the SIIM-ACR Pneumothorax
subset -- pairs each DICOM image with its decoded mask, resized to a
consistent size for batching.

Class imbalance note: even on a positive (pneumothorax-present) image, the
lesion region is typically a small fraction of total pixels -- most pixels
in a chest X-ray are background/other anatomy, not the region of interest.
Plain per-pixel cross-entropy would let a model get deceptively low loss by
predicting "all background". This dataset doesn't rebalance pixels itself
(that's the loss function's job, see Milestone 5's Dice+BCE combination) --
but it does keep a roughly even split of positive/negative *images* so the
model sees enough real lesions to learn from.
"""
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from .preprocessing import dicom_to_image
from .rle import rle_decode_multi

PathLike = str | Path


@dataclass
class ManifestEntry:
    uid: str
    dicom_path: Path
    rle_strings: List[str]
    has_mask: bool


def build_manifest(csv_path: PathLike, dicom_dir: PathLike) -> List[ManifestEntry]:
    """Pair each locally-downloaded DICOM file with its RLE mask string(s)
    from the labels CSV, keyed by SOPInstanceUID (the DICOM filename stem).

    Only images that actually exist in `dicom_dir` are included, since the
    subset downloaded to disk is a fraction of the full labels CSV.
    """
    dicom_dir = Path(dicom_dir)
    df = pd.read_csv(csv_path)
    df["EncodedPixels"] = df["EncodedPixels"].astype(str).str.strip()

    grouped = df.groupby("ImageId")["EncodedPixels"].apply(list)

    entries = []
    for dicom_path in sorted(dicom_dir.glob("*.dcm")):
        uid = dicom_path.stem
        if uid not in grouped.index:
            continue
        rle_strings = grouped.loc[uid]
        has_mask = any(s != "-1" for s in rle_strings)
        entries.append(ManifestEntry(uid=uid, dicom_path=dicom_path,
                                      rle_strings=rle_strings, has_mask=has_mask))
    return entries


def train_val_split(entries: List[ManifestEntry], val_fraction: float = 0.2,
                     seed: int = 42) -> tuple[List[ManifestEntry], List[ManifestEntry]]:
    """Stratified split by has_mask, so both splits keep a similar
    positive/negative ratio.
    """
    rng = np.random.default_rng(seed)
    positives = [e for e in entries if e.has_mask]
    negatives = [e for e in entries if not e.has_mask]

    def split_group(group):
        idx = rng.permutation(len(group))
        n_val = max(1, int(len(group) * val_fraction)) if group else 0
        val_idx = set(idx[:n_val].tolist())
        train = [g for i, g in enumerate(group) if i not in val_idx]
        val = [g for i, g in enumerate(group) if i in val_idx]
        return train, val

    train_pos, val_pos = split_group(positives)
    train_neg, val_neg = split_group(negatives)
    return train_pos + train_neg, val_pos + val_neg


class SIIMPneumothoraxDataset(Dataset):
    """Returns (image_tensor, mask_tensor), both float32 [1, H, W] in
    [0, 1] / {0, 1}, resized to `target_size`.
    """

    def __init__(self, entries: List[ManifestEntry], target_size: int = 256):
        self.entries = entries
        self.target_size = target_size

    def __len__(self) -> int:
        return len(self.entries)

    def __getitem__(self, idx: int):
        entry = self.entries[idx]

        image = dicom_to_image(entry.dicom_path, output_range="unit")  # (H, W) float32
        mask = rle_decode_multi(entry.rle_strings, height=image.shape[0], width=image.shape[1])

        size = (self.target_size, self.target_size)
        image_resized = cv2.resize(image, size, interpolation=cv2.INTER_AREA)
        # Nearest-neighbor for the mask: any smoothing interpolation would
        # produce fractional (non-binary) mask values at edges.
        mask_resized = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)

        image_tensor = torch.from_numpy(image_resized).float().unsqueeze(0)
        mask_tensor = torch.from_numpy(mask_resized).float().unsqueeze(0)
        return image_tensor, mask_tensor


In [ ]:
%%writefile dicom_seg/unet.py
"""
Milestone 4: U-Net for binary segmentation, built from scratch (Ronneberger
et al., 2015).

Encoder: repeated (Conv-BN-ReLU) x2 blocks, each followed by MaxPool2d to
halve spatial resolution while doubling channel depth.
Bottleneck: the deepest, lowest-resolution block.
Decoder: ConvTranspose2d doubles resolution back up at each stage. Before
each decoder conv block, the corresponding encoder feature map (same
resolution) is concatenated on the channel axis -- the skip connection.
Skip connections are what let the decoder recover fine spatial detail that
plain downsampling destroys; without them the output is a blurry,
low-resolution guess upsampled back to full size, not a precise mask.
Output: 1x1 conv to a single channel + sigmoid, for binary segmentation.
"""
from __future__ import annotations

from typing import List

import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    """Conv2d -> BatchNorm -> ReLU, twice."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UNet(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 1,
        features: List[int] = (64, 128, 256, 512),
    ):
        super().__init__()
        features = list(features)

        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        prev_channels = in_channels
        for f in features:
            self.encoders.append(ConvBlock(prev_channels, f))
            self.pools.append(nn.MaxPool2d(kernel_size=2, stride=2))
            prev_channels = f

        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)

        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        prev_channels = features[-1] * 2
        for f in reversed(features):
            self.upconvs.append(
                nn.ConvTranspose2d(prev_channels, f, kernel_size=2, stride=2)
            )
            # input channels doubled: upsampled features concatenated with
            # the skip connection from the corresponding encoder stage.
            self.decoders.append(ConvBlock(f * 2, f))
            prev_channels = f

        self.output_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        self.output_activation = nn.Sigmoid()

    def _logits(self, x: torch.Tensor) -> torch.Tensor:
        skip_connections = []
        for encoder, pool in zip(self.encoders, self.pools):
            x = encoder(x)
            skip_connections.append(x)
            x = pool(x)

        x = self.bottleneck(x)

        for upconv, decoder, skip in zip(
            self.upconvs, self.decoders, reversed(skip_connections)
        ):
            x = upconv(x)
            if x.shape[-2:] != skip.shape[-2:]:
                # Only triggered by odd input sizes that don't divide evenly
                # by 2 at every downsampling stage; crop the skip connection
                # to match rather than silently misaligning features.
                skip = _center_crop(skip, x.shape[-2:])
            x = torch.cat([skip, x], dim=1)
            x = decoder(x)

        return self.output_conv(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.output_activation(self._logits(x))

    def forward_with_logits(self, x: torch.Tensor):
        """Same forward pass, but also returns the pre-sigmoid logits --
        needed by Seg-Grad-CAM, which backpropagates from raw logits rather
        than the saturating sigmoid output.
        """
        logits = self._logits(x)
        return logits, self.output_activation(logits)


def _center_crop(tensor: torch.Tensor, target_hw) -> torch.Tensor:
    _, _, h, w = tensor.shape
    target_h, target_w = target_hw
    top = (h - target_h) // 2
    left = (w - target_w) // 2
    return tensor[:, :, top:top + target_h, left:left + target_w]


In [ ]:
%%writefile dicom_seg/losses.py
"""
Milestone 5: combined Dice + BCE loss.

Plain BCE treats every pixel independently. In this dataset most pixels
are background (see the class-imbalance note in dataset.py) -- a model can
reach deceptively low BCE while predicting an almost-empty mask, because
getting ~95% of (mostly background) pixels "right" by predicting nothing
still scores well pixel-by-pixel. Dice loss instead directly measures
overlap between predicted and true mask (2*|A∩B| / (|A|+|B|)), which stays
informative even when the foreground region is small relative to the
image. Combining the two gives Dice's region-overlap signal plus BCE's
stable per-pixel gradient, which is the standard recipe for imbalanced
binary segmentation.
"""
from __future__ import annotations

import torch
import torch.nn as nn


def dice_loss(pred: torch.Tensor, target: torch.Tensor, smooth: float = 1e-6) -> torch.Tensor:
    """Soft Dice loss. `pred` is expected to already be sigmoid
    probabilities in [0, 1] (not logits); `target` is a 0/1 mask. Both
    shaped [B, C, H, W] (or broadcastable). Averaged over the batch.
    """
    pred_flat = pred.reshape(pred.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1)

    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()


class DiceBCELoss(nn.Module):
    """loss = bce_weight * BCE(pred, target) + dice_weight * DiceLoss(pred, target)"""

    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5, smooth: float = 1e-6):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.bce = nn.BCELoss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce_loss = self.bce(pred, target)
        d_loss = dice_loss(pred, target, smooth=self.smooth)
        return self.bce_weight * bce_loss + self.dice_weight * d_loss


In [ ]:
%%writefile dicom_seg/metrics.py
"""
Milestone 6: Dice coefficient and IoU (Jaccard index) for binary
segmentation evaluation -- the "hard" (thresholded) counterparts to the
soft Dice loss used for training.

Dice = 2*|A∩B| / (|A|+|B|)   (equivalent to the F1 score for segmentation)
IoU  = |A∩B| / |A∪B|          (stricter: penalizes false positives/negatives
                                more than Dice does, since the denominator
                                doesn't double-count the intersection)
"""
from __future__ import annotations

import numpy as np
import torch


def _to_binary_numpy(mask, threshold: float = 0.5) -> np.ndarray:
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy()
    mask = np.asarray(mask)
    return (mask >= threshold).astype(np.uint8)


def dice_coefficient(pred, target, threshold: float = 0.5, smooth: float = 1e-6) -> float:
    """`pred` may be probabilities (thresholded at `threshold`) or already
    a 0/1 mask. `target` is a 0/1 (or already-binary) mask. Computed over
    all elements passed in (call per-sample for a per-image score).
    """
    pred_bin = _to_binary_numpy(pred, threshold)
    target_bin = _to_binary_numpy(target, threshold=0.5)

    intersection = np.logical_and(pred_bin, target_bin).sum()
    total = pred_bin.sum() + target_bin.sum()
    if total == 0:
        return 1.0  # both empty: perfect agreement, not undefined
    return float((2.0 * intersection + smooth) / (total + smooth))


def iou_score(pred, target, threshold: float = 0.5, smooth: float = 1e-6) -> float:
    pred_bin = _to_binary_numpy(pred, threshold)
    target_bin = _to_binary_numpy(target, threshold=0.5)

    intersection = np.logical_and(pred_bin, target_bin).sum()
    union = np.logical_or(pred_bin, target_bin).sum()
    if union == 0:
        return 1.0  # both empty: perfect agreement, not undefined
    return float((intersection + smooth) / (union + smooth))


In [ ]:
%%writefile dicom_seg/baseline.py
"""
Milestone 6: a naive, no-learning baseline to compare the U-Net against.

Otsu's method picks a single global intensity threshold that best
separates an image into two classes, with no training and no notion of
anatomy. Pneumothorax (air trapped outside a collapsed lung) shows up as an
abnormally lucent (dark) region with no lung markings, so "flag the darker
tissue-density pixels" is at least a plausible naive heuristic -- not a
strawman like "predict nothing" -- while still being something a first
image-processing course would produce, with no learned structure at all.
"""
from __future__ import annotations

import cv2
import numpy as np


def naive_threshold_baseline(image: np.ndarray) -> np.ndarray:
    """`image` is a float32 array in [0, 1] (as returned by dicom_to_image).
    Returns a 0/1 uint8 mask flagging pixels darker than the image's Otsu
    threshold, excluding pure-black background (outside the body) so the
    baseline isn't trivially "everything outside the chest".
    """
    img_u8 = (np.clip(image, 0.0, 1.0) * 255).astype(np.uint8)
    otsu_value, _ = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (img_u8 > 0) & (img_u8 <= otsu_value)
    return mask.astype(np.uint8)


In [ ]:
%%writefile dicom_seg/gradcam.py
"""
Milestone 7: Seg-Grad-CAM -- Grad-CAM adapted for segmentation.

Grad-CAM was originally defined for classification: backpropagate a single
predicted class *score* to a convolutional layer's activations, weight each
channel by its averaged gradient (how much that channel's activation
influenced the score), sum, and ReLU (only positive influence matters).

Segmentation has no single class score. The adaptation used here (a
straightforward reading of what's often called Seg-Grad-CAM) replaces the
classification score with the sum of the predicted mask's LOGITS over a
region of interest -- by default the whole image, or a caller-supplied
binary region to explain "why did the model activate *here*". Logits are
used rather than the post-sigmoid probabilities because sigmoid saturates:
gradients through a probability near 0 or 1 vanish, which would make the
attribution unreliable exactly where the model is most confident.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F


class SegGradCAM:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        """`target_layer` is typically the last conv layer of the encoder
        or the bottleneck -- deep enough to encode high-level structure,
        shallow enough (spatially) to still localize.
        """
        self.model = model
        self.target_layer = target_layer
        self._activations = None
        self._gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self._activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self._gradients = grad_output[0].detach()

    def __call__(self, input_tensor: torch.Tensor, region: torch.Tensor | None = None):
        """Returns (cam, probs): `cam` is a [H, W] numpy heatmap in [0, 1]
        for input_tensor[0] (batch size 1 expected), upsampled to the
        input's spatial size. `probs` is the model's normal sigmoid output.
        """
        self.model.zero_grad(set_to_none=True)
        logits, probs = self.model.forward_with_logits(input_tensor)

        score = (logits * region).sum() if region is not None else logits.sum()
        score.backward()

        if self._gradients is None or self._activations is None:
            raise RuntimeError("target_layer hooks never fired -- is it part of the model's forward pass?")

        # Global-average-pool the gradients over space -> one importance
        # weight per channel, exactly as in the original Grad-CAM.
        weights = self._gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self._activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)  # only positive influence on the score matters

        cam = F.interpolate(
            cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False
        )
        cam = cam[0, 0]
        cam_min, cam_max = cam.min(), cam.max()
        if (cam_max - cam_min) > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = torch.zeros_like(cam)

        return cam.cpu().numpy(), probs.detach()


## Download the same 400-image subset used locally

Same UID selection (seed 42, 200 positive / 200 negative) as `data/siim_subset/selected_uids.json` in the local repo -- so the train/val split, and therefore the baseline comparison in `BENCHMARKS.md`, line up exactly.

In [ ]:
!mkdir -p data/siim_subset

In [ ]:
%%writefile data/selected_uids_compact.json
{"positive": ["1.2.276.0.7230010.3.1.4.8323329.11794.1517875234.871858", "1.2.276.0.7230010.3.1.4.8323329.10348.1517875223.298190", "1.2.276.0.7230010.3.1.4.8323329.1512.1517875168.175004", "1.2.276.0.7230010.3.1.4.8323329.14110.1517875249.673446", "1.2.276.0.7230010.3.1.4.8323329.13730.1517875247.610996", "1.2.276.0.7230010.3.1.4.8323329.1229.1517875166.872911", "1.2.276.0.7230010.3.1.4.8323329.11670.1517875234.243046", "1.2.276.0.7230010.3.1.4.8323329.5235.1517875187.117250", "1.2.276.0.7230010.3.1.4.8323329.1140.1517875166.465295", "1.2.276.0.7230010.3.1.4.8323329.32680.1517875161.762287", "1.2.276.0.7230010.3.1.4.8323329.10464.1517875223.829669", "1.2.276.0.7230010.3.1.4.8323329.10426.1517875223.671083", "1.2.276.0.7230010.3.1.4.8323329.11514.1517875233.284248", "1.2.276.0.7230010.3.1.4.8323329.13667.1517875247.125535", "1.2.276.0.7230010.3.1.4.8323329.13907.1517875248.593038", "1.2.276.0.7230010.3.1.4.8323329.4628.1517875183.816936", "1.2.276.0.7230010.3.1.4.8323329.10367.1517875223.390827", "1.2.276.0.7230010.3.1.4.8323329.550.1517875163.394533", "1.2.276.0.7230010.3.1.4.8323329.13294.1517875244.543387", "1.2.276.0.7230010.3.1.4.8323329.5231.1517875187.99671", "1.2.276.0.7230010.3.1.4.8323329.32639.1517875161.579169", "1.2.276.0.7230010.3.1.4.8323329.13684.1517875247.213718", "1.2.276.0.7230010.3.1.4.8323329.368.1517875162.584967", "1.2.276.0.7230010.3.1.4.8323329.1556.1517875168.390347", "1.2.276.0.7230010.3.1.4.8323329.10099.1517875222.78541", "1.2.276.0.7230010.3.1.4.8323329.1263.1517875167.19062", "1.2.276.0.7230010.3.1.4.8323329.32696.1517875161.855596", "1.2.276.0.7230010.3.1.4.8323329.2768.1517875174.607604", "1.2.276.0.7230010.3.1.4.8323329.1554.1517875168.395268", "1.2.276.0.7230010.3.1.4.8323329.12535.1517875239.599448", "1.2.276.0.7230010.3.1.4.8323329.13606.1517875246.843676", "1.2.276.0.7230010.3.1.4.8323329.2667.1517875173.957745", "1.2.276.0.7230010.3.1.4.8323329.11669.1517875234.229971", "1.2.276.0.7230010.3.1.4.8323329.11489.1517875233.189659", "1.2.276.0.7230010.3.1.4.8323329.31915.1517875157.480734", "1.2.276.0.7230010.3.1.4.8323329.11599.1517875233.848287", "1.2.276.0.7230010.3.1.4.8323329.3082.1517875175.983997", "1.2.276.0.7230010.3.1.4.8323329.2825.1517875174.908244", "1.2.276.0.7230010.3.1.4.8323329.14439.1517875252.132263", "1.2.276.0.7230010.3.1.4.8323329.10659.1517875224.931582", "1.2.276.0.7230010.3.1.4.8323329.3838.1517875179.729923", "1.2.276.0.7230010.3.1.4.8323329.506.1517875163.231648", "1.2.276.0.7230010.3.1.4.8323329.12018.1517875236.922009", "1.2.276.0.7230010.3.1.4.8323329.31892.1517875157.349249", "1.2.276.0.7230010.3.1.4.8323329.11246.1517875231.952753", "1.2.276.0.7230010.3.1.4.8323329.5358.1517875187.748481", "1.2.276.0.7230010.3.1.4.8323329.1824.1517875169.753287", "1.2.276.0.7230010.3.1.4.8323329.3123.1517875176.214943", "1.2.276.0.7230010.3.1.4.8323329.5747.1517875190.107803", "1.2.276.0.7230010.3.1.4.8323329.13177.1517875244.13038", "1.2.276.0.7230010.3.1.4.8323329.11083.1517875230.944434", "1.2.276.0.7230010.3.1.4.8323329.10691.1517875225.96424", "1.2.276.0.7230010.3.1.4.8323329.13801.1517875248.18204", "1.2.276.0.7230010.3.1.4.8323329.1769.1517875169.341759", "1.2.276.0.7230010.3.1.4.8323329.11275.1517875232.96765", "1.2.276.0.7230010.3.1.4.8323329.13911.1517875248.626599", "1.2.276.0.7230010.3.1.4.8323329.11657.1517875234.172462", "1.2.276.0.7230010.3.1.4.8323329.3743.1517875179.246355", "1.2.276.0.7230010.3.1.4.8323329.3155.1517875176.356160", "1.2.276.0.7230010.3.1.4.8323329.12693.1517875241.217666", "1.2.276.0.7230010.3.1.4.8323329.31753.1517875156.623360", "1.2.276.0.7230010.3.1.4.8323329.3028.1517875175.711269", "1.2.276.0.7230010.3.1.4.8323329.13513.1517875246.27434", "1.2.276.0.7230010.3.1.4.8323329.14496.1517875252.386912", "1.2.276.0.7230010.3.1.4.8323329.11139.1517875231.356380", "1.2.276.0.7230010.3.1.4.8323329.12820.1517875242.12046", "1.2.276.0.7230010.3.1.4.8323329.5020.1517875185.984691", "1.2.276.0.7230010.3.1.4.8323329.14099.1517875249.625073", "1.2.276.0.7230010.3.1.4.8323329.12701.1517875241.233681", "1.2.276.0.7230010.3.1.4.8323329.3875.1517875179.932640", "1.2.276.0.7230010.3.1.4.8323329.31905.1517875157.437107", "1.2.276.0.7230010.3.1.4.8323329.14559.1517875252.701386", "1.2.276.0.7230010.3.1.4.8323329.5423.1517875188.37987", "1.2.276.0.7230010.3.1.4.8323329.13676.1517875247.165344", "1.2.276.0.7230010.3.1.4.8323329.2459.1517875172.853488", "1.2.276.0.7230010.3.1.4.8323329.10884.1517875226.103022", "1.2.276.0.7230010.3.1.4.8323329.13831.1517875248.167238", "1.2.276.0.7230010.3.1.4.8323329.10466.1517875223.830304", "1.2.276.0.7230010.3.1.4.8323329.2294.1517875171.978170", "1.2.276.0.7230010.3.1.4.8323329.32281.1517875159.350469", "1.2.276.0.7230010.3.1.4.8323329.14508.1517875252.443873", "1.2.276.0.7230010.3.1.4.8323329.11031.1517875230.4520", "1.2.276.0.7230010.3.1.4.8323329.13540.1517875246.540854", "1.2.276.0.7230010.3.1.4.8323329.559.1517875163.427432", "1.2.276.0.7230010.3.1.4.8323329.2265.1517875171.868079", "1.2.276.0.7230010.3.1.4.8323329.13564.1517875246.632212", "1.2.276.0.7230010.3.1.4.8323329.4520.1517875183.276045", "1.2.276.0.7230010.3.1.4.8323329.32194.1517875158.962626", "1.2.276.0.7230010.3.1.4.8323329.3815.1517875179.613404", "1.2.276.0.7230010.3.1.4.8323329.12348.1517875238.700620", "1.2.276.0.7230010.3.1.4.8323329.14449.1517875252.165529", "1.2.276.0.7230010.3.1.4.8323329.1414.1517875167.654443", "1.2.276.0.7230010.3.1.4.8323329.5502.1517875188.537175", "1.2.276.0.7230010.3.1.4.8323329.5105.1517875186.423825", "1.2.276.0.7230010.3.1.4.8323329.14413.1517875251.979819", "1.2.276.0.7230010.3.1.4.8323329.32762.1517875162.235692", "1.2.276.0.7230010.3.1.4.8323329.32254.1517875159.256888", "1.2.276.0.7230010.3.1.4.8323329.3125.1517875176.223801", "1.2.276.0.7230010.3.1.4.8323329.13671.1517875247.141784", "1.2.276.0.7230010.3.1.4.8323329.12267.1517875238.313953", "1.2.276.0.7230010.3.1.4.8323329.4693.1517875184.149553", "1.2.276.0.7230010.3.1.4.8323329.4435.1517875182.866786", "1.2.276.0.7230010.3.1.4.8323329.11462.1517875233.57314", "1.2.276.0.7230010.3.1.4.8323329.10725.1517875225.308470", "1.2.276.0.7230010.3.1.4.8323329.11777.1517875234.742626", "1.2.276.0.7230010.3.1.4.8323329.12494.1517875239.387262", "1.2.276.0.7230010.3.1.4.8323329.12640.1517875240.944081", "1.2.276.0.7230010.3.1.4.8323329.32681.1517875161.766764", "1.2.276.0.7230010.3.1.4.8323329.11002.1517875229.349676", "1.2.276.0.7230010.3.1.4.8323329.3201.1517875176.562886", "1.2.276.0.7230010.3.1.4.8323329.31951.1517875157.659578", "1.2.276.0.7230010.3.1.4.8323329.3968.1517875180.410977", "1.2.276.0.7230010.3.1.4.8323329.4961.1517875185.712498", "1.2.276.0.7230010.3.1.4.8323329.14213.1517875250.265210", "1.2.276.0.7230010.3.1.4.8323329.5377.1517875187.826376", "1.2.276.0.7230010.3.1.4.8323329.1017.1517875165.920291", "1.2.276.0.7230010.3.1.4.8323329.11861.1517875236.75447", "1.2.276.0.7230010.3.1.4.8323329.5070.1517875186.277826", "1.2.276.0.7230010.3.1.4.8323329.14482.1517875252.317331", "1.2.276.0.7230010.3.1.4.8323329.1838.1517875169.866736", "1.2.276.0.7230010.3.1.4.8323329.3429.1517875177.805141", "1.2.276.0.7230010.3.1.4.8323329.12587.1517875239.902166", "1.2.276.0.7230010.3.1.4.8323329.3745.1517875179.256119", "1.2.276.0.7230010.3.1.4.8323329.10050.1517875221.763035", "1.2.276.0.7230010.3.1.4.8323329.14415.1517875251.998778", "1.2.276.0.7230010.3.1.4.8323329.4536.1517875183.345270", "1.2.276.0.7230010.3.1.4.8323329.12952.1517875242.695935", "1.2.276.0.7230010.3.1.4.8323329.4665.1517875184.34018", "1.2.276.0.7230010.3.1.4.8323329.11731.1517875234.510759", "1.2.276.0.7230010.3.1.4.8323329.1952.1517875170.398574", "1.2.276.0.7230010.3.1.4.8323329.31814.1517875157.8691", "1.2.276.0.7230010.3.1.4.8323329.12671.1517875241.63757", "1.2.276.0.7230010.3.1.4.8323329.5109.1517875186.450605", "1.2.276.0.7230010.3.1.4.8323329.4974.1517875185.796001", "1.2.276.0.7230010.3.1.4.8323329.10012.1517875220.965942", "1.2.276.0.7230010.3.1.4.8323329.2457.1517875172.834411", "1.2.276.0.7230010.3.1.4.8323329.4362.1517875182.469446", "1.2.276.0.7230010.3.1.4.8323329.10269.1517875222.913699", "1.2.276.0.7230010.3.1.4.8323329.11804.1517875234.959778", "1.2.276.0.7230010.3.1.4.8323329.3134.1517875176.262695", "1.2.276.0.7230010.3.1.4.8323329.2147.1517875171.360386", "1.2.276.0.7230010.3.1.4.8323329.14018.1517875249.164341", "1.2.276.0.7230010.3.1.4.8323329.10911.1517875227.881792", "1.2.276.0.7230010.3.1.4.8323329.14032.1517875249.239494", "1.2.276.0.7230010.3.1.4.8323329.1138.1517875166.441012", "1.2.276.0.7230010.3.1.4.8323329.4310.1517875182.219135", "1.2.276.0.7230010.3.1.4.8323329.11075.1517875230.895962", "1.2.276.0.7230010.3.1.4.8323329.4997.1517875185.898581", "1.2.276.0.7230010.3.1.4.8323329.12039.1517875237.59798", "1.2.276.0.7230010.3.1.4.8323329.1207.1517875166.781091", "1.2.276.0.7230010.3.1.4.8323329.4146.1517875181.343714", "1.2.276.0.7230010.3.1.4.8323329.5317.1517875187.580549", "1.2.276.0.7230010.3.1.4.8323329.12724.1517875241.370386", "1.2.276.0.7230010.3.1.4.8323329.14453.1517875252.188348", "1.2.276.0.7230010.3.1.4.8323329.4937.1517875185.552993", "1.2.276.0.7230010.3.1.4.8323329.327.1517875162.393558", "1.2.276.0.7230010.3.1.4.8323329.13559.1517875246.622712", "1.2.276.0.7230010.3.1.4.8323329.5108.1517875186.444900", "1.2.276.0.7230010.3.1.4.8323329.13330.1517875244.685390", "1.2.276.0.7230010.3.1.4.8323329.2218.1517875171.672924", "1.2.276.0.7230010.3.1.4.8323329.32252.1517875159.238434", "1.2.276.0.7230010.3.1.4.8323329.31805.1517875156.946483", "1.2.276.0.7230010.3.1.4.8323329.3475.1517875178.11262", "1.2.276.0.7230010.3.1.4.8323329.4794.1517875184.840760", "1.2.276.0.7230010.3.1.4.8323329.3711.1517875179.120533", "1.2.276.0.7230010.3.1.4.8323329.11959.1517875236.636622", "1.2.276.0.7230010.3.1.4.8323329.14170.1517875250.5354", "1.2.276.0.7230010.3.1.4.8323329.13749.1517875247.714088", "1.2.276.0.7230010.3.1.4.8323329.11008.1517875229.467452", "1.2.276.0.7230010.3.1.4.8323329.2713.1517875174.188646", "1.2.276.0.7230010.3.1.4.8323329.10295.1517875223.23166", "1.2.276.0.7230010.3.1.4.8323329.5381.1517875187.848196", "1.2.276.0.7230010.3.1.4.8323329.13861.1517875248.361182", "1.2.276.0.7230010.3.1.4.8323329.10107.1517875222.121330", "1.2.276.0.7230010.3.1.4.8323329.11135.1517875231.335673", "1.2.276.0.7230010.3.1.4.8323329.10918.1517875227.904054", "1.2.276.0.7230010.3.1.4.8323329.13829.1517875248.155435", "1.2.276.0.7230010.3.1.4.8323329.11056.1517875230.475630", "1.2.276.0.7230010.3.1.4.8323329.10452.1517875223.788338", "1.2.276.0.7230010.3.1.4.8323329.2579.1517875173.586381", "1.2.276.0.7230010.3.1.4.8323329.4741.1517875184.511107", "1.2.276.0.7230010.3.1.4.8323329.13996.1517875249.45103", "1.2.276.0.7230010.3.1.4.8323329.1557.1517875168.394768", "1.2.276.0.7230010.3.1.4.8323329.4297.1517875182.164262", "1.2.276.0.7230010.3.1.4.8323329.13594.1517875246.789265", "1.2.276.0.7230010.3.1.4.8323329.12162.1517875237.788063", "1.2.276.0.7230010.3.1.4.8323329.5650.1517875189.325122", "1.2.276.0.7230010.3.1.4.8323329.5741.1517875190.56470", "1.2.276.0.7230010.3.1.4.8323329.4084.1517875181.28386", "1.2.276.0.7230010.3.1.4.8323329.14051.1517875249.351067", "1.2.276.0.7230010.3.1.4.8323329.4095.1517875181.100671", "1.2.276.0.7230010.3.1.4.8323329.32389.1517875160.347239", "1.2.276.0.7230010.3.1.4.8323329.13128.1517875243.795794", "1.2.276.0.7230010.3.1.4.8323329.11538.1517875233.464937", "1.2.276.0.7230010.3.1.4.8323329.11601.1517875233.863251", "1.2.276.0.7230010.3.1.4.8323329.3334.1517875177.309082", "1.2.276.0.7230010.3.1.4.8323329.3017.1517875175.684558", "1.2.276.0.7230010.3.1.4.8323329.32701.1517875161.857426", "1.2.276.0.7230010.3.1.4.8323329.32471.1517875160.769925", "1.2.276.0.7230010.3.1.4.8323329.3960.1517875180.358445"], "negative": ["1.2.276.0.7230010.3.1.4.8323329.11072.1517875230.888179", "1.2.276.0.7230010.3.1.4.8323329.11934.1517875236.540661", "1.2.276.0.7230010.3.1.4.8323329.11199.1517875231.672005", "1.2.276.0.7230010.3.1.4.8323329.377.1517875162.629482", "1.2.276.0.7230010.3.1.4.8323329.32050.1517875158.202694", "1.2.276.0.7230010.3.1.4.8323329.12140.1517875237.634506", "1.2.276.0.7230010.3.1.4.8323329.1676.1517875168.871500", "1.2.276.0.7230010.3.1.4.8323329.13721.1517875247.380606", "1.2.276.0.7230010.3.1.4.8323329.13692.1517875247.258127", "1.2.276.0.7230010.3.1.4.8323329.4647.1517875183.972236", "1.2.276.0.7230010.3.1.4.8323329.12730.1517875241.539167", "1.2.276.0.7230010.3.1.4.8323329.4128.1517875181.238960", "1.2.276.0.7230010.3.1.4.8323329.13556.1517875246.601809", "1.2.276.0.7230010.3.1.4.8323329.2297.1517875171.995032", "1.2.276.0.7230010.3.1.4.8323329.4938.1517875185.561211", "1.2.276.0.7230010.3.1.4.8323329.1701.1517875169.20014", "1.2.276.0.7230010.3.1.4.8323329.1148.1517875166.497982", "1.2.276.0.7230010.3.1.4.8323329.4548.1517875183.395700", "1.2.276.0.7230010.3.1.4.8323329.11923.1517875236.496214", "1.2.276.0.7230010.3.1.4.8323329.10997.1517875229.282549", "1.2.276.0.7230010.3.1.4.8323329.10302.1517875223.55541", "1.2.276.0.7230010.3.1.4.8323329.11832.1517875235.929649", "1.2.276.0.7230010.3.1.4.8323329.14567.1517875252.737433", "1.2.276.0.7230010.3.1.4.8323329.13237.1517875244.282346", "1.2.276.0.7230010.3.1.4.8323329.384.1517875162.674411", "1.2.276.0.7230010.3.1.4.8323329.5388.1517875187.890892", "1.2.276.0.7230010.3.1.4.8323329.5301.1517875187.499884", "1.2.276.0.7230010.3.1.4.8323329.14140.1517875249.807509", "1.2.276.0.7230010.3.1.4.8323329.3734.1517875179.197790", "1.2.276.0.7230010.3.1.4.8323329.11160.1517875231.459962", "1.2.276.0.7230010.3.1.4.8323329.13203.1517875244.117978", "1.2.276.0.7230010.3.1.4.8323329.3314.1517875177.207552", "1.2.276.0.7230010.3.1.4.8323329.10041.1517875221.697952", "1.2.276.0.7230010.3.1.4.8323329.3535.1517875178.271143", "1.2.276.0.7230010.3.1.4.8323329.2020.1517875170.725653", "1.2.276.0.7230010.3.1.4.8323329.4781.1517875184.743287", "1.2.276.0.7230010.3.1.4.8323329.2436.1517875172.725410", "1.2.276.0.7230010.3.1.4.8323329.4149.1517875181.364853", "1.2.276.0.7230010.3.1.4.8323329.5407.1517875187.957019", "1.2.276.0.7230010.3.1.4.8323329.13014.1517875243.90817", "1.2.276.0.7230010.3.1.4.8323329.13688.1517875247.231744", "1.2.276.0.7230010.3.1.4.8323329.2694.1517875174.84554", "1.2.276.0.7230010.3.1.4.8323329.14217.1517875250.282721", "1.2.276.0.7230010.3.1.4.8323329.11156.1517875231.446627", "1.2.276.0.7230010.3.1.4.8323329.11203.1517875231.691174", "1.2.276.0.7230010.3.1.4.8323329.3081.1517875175.964702", "1.2.276.0.7230010.3.1.4.8323329.11126.1517875231.302385", "1.2.276.0.7230010.3.1.4.8323329.1099.1517875166.314896", "1.2.276.0.7230010.3.1.4.8323329.5212.1517875187.5789", "1.2.276.0.7230010.3.1.4.8323329.5723.1517875189.764583", "1.2.276.0.7230010.3.1.4.8323329.13068.1517875243.399267", "1.2.276.0.7230010.3.1.4.8323329.11121.1517875231.258211", "1.2.276.0.7230010.3.1.4.8323329.11565.1517875233.635888", "1.2.276.0.7230010.3.1.4.8323329.13604.1517875246.827790", "1.2.276.0.7230010.3.1.4.8323329.1135.1517875166.429148", "1.2.276.0.7230010.3.1.4.8323329.11338.1517875232.452485", "1.2.276.0.7230010.3.1.4.8323329.14546.1517875252.633721", "1.2.276.0.7230010.3.1.4.8323329.3793.1517875179.441128", "1.2.276.0.7230010.3.1.4.8323329.12343.1517875238.674757", "1.2.276.0.7230010.3.1.4.8323329.1622.1517875168.630031", "1.2.276.0.7230010.3.1.4.8323329.10784.1517875225.617068", "1.2.276.0.7230010.3.1.4.8323329.11600.1517875233.854116", "1.2.276.0.7230010.3.1.4.8323329.4079.1517875181.3810", "1.2.276.0.7230010.3.1.4.8323329.3145.1517875176.315654", "1.2.276.0.7230010.3.1.4.8323329.1930.1517875170.305802", "1.2.276.0.7230010.3.1.4.8323329.13960.1517875248.853341", "1.2.276.0.7230010.3.1.4.8323329.3095.1517875176.71736", "1.2.276.0.7230010.3.1.4.8323329.14609.1517875253.57642", "1.2.276.0.7230010.3.1.4.8323329.2028.1517875170.776571", "1.2.276.0.7230010.3.1.4.8323329.3633.1517875178.761935", "1.2.276.0.7230010.3.1.4.8323329.12562.1517875239.738012", "1.2.276.0.7230010.3.1.4.8323329.2759.1517875174.578599", "1.2.276.0.7230010.3.1.4.8323329.4836.1517875185.49416", "1.2.276.0.7230010.3.1.4.8323329.3142.1517875176.298724", "1.2.276.0.7230010.3.1.4.8323329.11424.1517875232.878433", "1.2.276.0.7230010.3.1.4.8323329.10189.1517875222.538484", "1.2.276.0.7230010.3.1.4.8323329.4856.1517875185.126181", "1.2.276.0.7230010.3.1.4.8323329.11964.1517875236.659104", "1.2.276.0.7230010.3.1.4.8323329.1144.1517875166.464567", "1.2.276.0.7230010.3.1.4.8323329.14129.1517875249.758591", "1.2.276.0.7230010.3.1.4.8323329.5788.1517875190.602701", "1.2.276.0.7230010.3.1.4.8323329.12590.1517875239.924542", "1.2.276.0.7230010.3.1.4.8323329.32236.1517875159.175562", "1.2.276.0.7230010.3.1.4.8323329.11354.1517875232.557775", "1.2.276.0.7230010.3.1.4.8323329.1582.1517875168.499039", "1.2.276.0.7230010.3.1.4.8323329.32623.1517875161.537934", "1.2.276.0.7230010.3.1.4.8323329.2430.1517875172.672388", "1.2.276.0.7230010.3.1.4.8323329.13076.1517875243.444620", "1.2.276.0.7230010.3.1.4.8323329.4449.1517875182.911064", "1.2.276.0.7230010.3.1.4.8323329.2815.1517875174.864775", "1.2.276.0.7230010.3.1.4.8323329.1016.1517875165.922067", "1.2.276.0.7230010.3.1.4.8323329.2749.1517875174.450094", "1.2.276.0.7230010.3.1.4.8323329.12035.1517875237.37522", "1.2.276.0.7230010.3.1.4.8323329.12620.1517875240.181236", "1.2.276.0.7230010.3.1.4.8323329.2007.1517875170.668750", "1.2.276.0.7230010.3.1.4.8323329.1226.1517875166.862141", "1.2.276.0.7230010.3.1.4.8323329.12102.1517875237.394438", "1.2.276.0.7230010.3.1.4.8323329.13028.1517875243.196056", "1.2.276.0.7230010.3.1.4.8323329.2164.1517875171.435571", "1.2.276.0.7230010.3.1.4.8323329.2363.1517875172.359831", "1.2.276.0.7230010.3.1.4.8323329.14081.1517875249.531983", "1.2.276.0.7230010.3.1.4.8323329.32117.1517875158.520071", "1.2.276.0.7230010.3.1.4.8323329.13948.1517875248.791860", "1.2.276.0.7230010.3.1.4.8323329.1998.1517875170.639555", "1.2.276.0.7230010.3.1.4.8323329.5775.1517875190.475246", "1.2.276.0.7230010.3.1.4.8323329.5445.1517875188.168800", "1.2.276.0.7230010.3.1.4.8323329.1727.1517875169.127192", "1.2.276.0.7230010.3.1.4.8323329.11001.1517875229.350228", "1.2.276.0.7230010.3.1.4.8323329.11816.1517875235.159060", "1.2.276.0.7230010.3.1.4.8323329.4160.1517875181.431436", "1.2.276.0.7230010.3.1.4.8323329.2260.1517875171.857157", "1.2.276.0.7230010.3.1.4.8323329.10869.1517875226.41725", "1.2.276.0.7230010.3.1.4.8323329.10071.1517875221.840835", "1.2.276.0.7230010.3.1.4.8323329.31946.1517875157.626485", "1.2.276.0.7230010.3.1.4.8323329.12561.1517875239.724046", "1.2.276.0.7230010.3.1.4.8323329.1950.1517875170.385827", "1.2.276.0.7230010.3.1.4.8323329.13148.1517875243.884043", "1.2.276.0.7230010.3.1.4.8323329.4522.1517875183.294486", "1.2.276.0.7230010.3.1.4.8323329.4247.1517875181.881476", "1.2.276.0.7230010.3.1.4.8323329.1020.1517875165.935324", "1.2.276.0.7230010.3.1.4.8323329.1219.1517875166.835208", "1.2.276.0.7230010.3.1.4.8323329.11475.1517875233.110584", "1.2.276.0.7230010.3.1.4.8323329.12905.1517875242.471328", "1.2.276.0.7230010.3.1.4.8323329.1072.1517875166.176122", "1.2.276.0.7230010.3.1.4.8323329.32617.1517875161.501139", "1.2.276.0.7230010.3.1.4.8323329.12886.1517875242.371079", "1.2.276.0.7230010.3.1.4.8323329.4286.1517875182.128129", "1.2.276.0.7230010.3.1.4.8323329.12497.1517875239.403769", "1.2.276.0.7230010.3.1.4.8323329.10825.1517875225.796519", "1.2.276.0.7230010.3.1.4.8323329.2969.1517875175.502019", "1.2.276.0.7230010.3.1.4.8323329.32536.1517875161.68580", "1.2.276.0.7230010.3.1.4.8323329.10787.1517875225.620766", "1.2.276.0.7230010.3.1.4.8323329.32403.1517875160.433574", "1.2.276.0.7230010.3.1.4.8323329.14073.1517875249.496332", "1.2.276.0.7230010.3.1.4.8323329.1694.1517875168.978681", "1.2.276.0.7230010.3.1.4.8323329.12021.1517875236.944561", "1.2.276.0.7230010.3.1.4.8323329.32329.1517875160.91430", "1.2.276.0.7230010.3.1.4.8323329.13010.1517875243.42667", "1.2.276.0.7230010.3.1.4.8323329.14572.1517875252.770179", "1.2.276.0.7230010.3.1.4.8323329.13163.1517875243.948073", "1.2.276.0.7230010.3.1.4.8323329.13437.1517875245.324187", "1.2.276.0.7230010.3.1.4.8323329.3954.1517875180.312604", "1.2.276.0.7230010.3.1.4.8323329.10501.1517875223.977887", "1.2.276.0.7230010.3.1.4.8323329.13480.1517875245.508760", "1.2.276.0.7230010.3.1.4.8323329.31922.1517875157.507737", "1.2.276.0.7230010.3.1.4.8323329.3945.1517875180.257032", "1.2.276.0.7230010.3.1.4.8323329.1666.1517875168.835418", "1.2.276.0.7230010.3.1.4.8323329.2053.1517875170.918662", "1.2.276.0.7230010.3.1.4.8323329.13104.1517875243.628352", "1.2.276.0.7230010.3.1.4.8323329.12121.1517875237.479998", "1.2.276.0.7230010.3.1.4.8323329.3379.1517875177.565413", "1.2.276.0.7230010.3.1.4.8323329.10766.1517875225.551647", "1.2.276.0.7230010.3.1.4.8323329.5101.1517875186.411934", "1.2.276.0.7230010.3.1.4.8323329.14309.1517875250.708089", "1.2.276.0.7230010.3.1.4.8323329.13867.1517875248.389781", "1.2.276.0.7230010.3.1.4.8323329.4897.1517875185.315627", "1.2.276.0.7230010.3.1.4.8323329.32248.1517875159.221374", "1.2.276.0.7230010.3.1.4.8323329.2881.1517875175.213570", "1.2.276.0.7230010.3.1.4.8323329.14403.1517875251.947454", "1.2.276.0.7230010.3.1.4.8323329.14319.1517875251.281792", "1.2.276.0.7230010.3.1.4.8323329.1048.1517875166.56276", "1.2.276.0.7230010.3.1.4.8323329.13750.1517875247.722805", "1.2.276.0.7230010.3.1.4.8323329.3682.1517875178.968825", "1.2.276.0.7230010.3.1.4.8323329.31847.1517875157.128821", "1.2.276.0.7230010.3.1.4.8323329.2298.1517875171.992665", "1.2.276.0.7230010.3.1.4.8323329.11365.1517875232.614724", "1.2.276.0.7230010.3.1.4.8323329.2308.1517875172.78694", "1.2.276.0.7230010.3.1.4.8323329.32279.1517875159.341380", "1.2.276.0.7230010.3.1.4.8323329.3708.1517875179.90734", "1.2.276.0.7230010.3.1.4.8323329.31900.1517875157.391804", "1.2.276.0.7230010.3.1.4.8323329.10554.1517875224.245718", "1.2.276.0.7230010.3.1.4.8323329.12258.1517875238.264802", "1.2.276.0.7230010.3.1.4.8323329.1937.1517875170.331435", "1.2.276.0.7230010.3.1.4.8323329.13466.1517875245.459483", "1.2.276.0.7230010.3.1.4.8323329.2025.1517875170.762186", "1.2.276.0.7230010.3.1.4.8323329.10758.1517875225.499504", "1.2.276.0.7230010.3.1.4.8323329.12126.1517875237.530954", "1.2.276.0.7230010.3.1.4.8323329.4374.1517875182.563252", "1.2.276.0.7230010.3.1.4.8323329.32170.1517875158.856439", "1.2.276.0.7230010.3.1.4.8323329.3084.1517875175.991018", "1.2.276.0.7230010.3.1.4.8323329.4412.1517875182.702504", "1.2.276.0.7230010.3.1.4.8323329.12262.1517875238.279624", "1.2.276.0.7230010.3.1.4.8323329.3423.1517875177.779723", "1.2.276.0.7230010.3.1.4.8323329.13690.1517875247.243990", "1.2.276.0.7230010.3.1.4.8323329.1802.1517875169.619424", "1.2.276.0.7230010.3.1.4.8323329.10874.1517875226.70346", "1.2.276.0.7230010.3.1.4.8323329.4405.1517875182.677055", "1.2.276.0.7230010.3.1.4.8323329.1003.1517875165.884550", "1.2.276.0.7230010.3.1.4.8323329.13822.1517875248.113941", "1.2.276.0.7230010.3.1.4.8323329.3253.1517875176.905250", "1.2.276.0.7230010.3.1.4.8323329.4314.1517875182.241606", "1.2.276.0.7230010.3.1.4.8323329.11374.1517875232.662027", "1.2.276.0.7230010.3.1.4.8323329.31883.1517875157.307443", "1.2.276.0.7230010.3.1.4.8323329.3091.1517875176.53567", "1.2.276.0.7230010.3.1.4.8323329.12441.1517875239.134492", "1.2.276.0.7230010.3.1.4.8323329.2765.1517875174.594000", "1.2.276.0.7230010.3.1.4.8323329.2993.1517875175.586359", "1.2.276.0.7230010.3.1.4.8323329.3882.1517875179.968105", "1.2.276.0.7230010.3.1.4.8323329.318.1517875162.378490", "1.2.276.0.7230010.3.1.4.8323329.3766.1517875179.336588"]}

In [ ]:
!kaggle competitions download -c siim-acr-pneumothorax-segmentation -f stage_2_train.csv -p data/siim_subset
!cd data/siim_subset && unzip -o -q stage_2_train.csv.zip && rm -f stage_2_train.csv.zip
print("Labels CSV ready.")

In [ ]:
import json
import time
from pathlib import Path

from kaggle.api.kaggle_api_extended import KaggleApi

DATASET = "jesperdramsch/siim-acr-pneumothorax-segmentation-data"

with open("data/selected_uids_compact.json") as f:
    sel = json.load(f)
selected_uids = set(sel["positive"] + sel["negative"])
print(f"Selected {len(selected_uids)} images to download.")

api = KaggleApi()
api.authenticate()

token = None
path_by_uid = {}
while True:
    for attempt in range(6):
        try:
            resp = api.dataset_list_files(DATASET, page_token=token, page_size=200)
            break
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"retry after {wait}s: {e}")
            time.sleep(wait)
    else:
        raise RuntimeError("failed to list dataset files after retries")
    for f in resp.files:
        if f.name.startswith("dicom-images-train/"):
            uid = f.name.rsplit("/", 1)[-1][:-4]
            if uid in selected_uids:
                path_by_uid[uid] = f.name
    token = resp.next_page_token
    if not token or len(path_by_uid) >= len(selected_uids):
        break
    time.sleep(1.0)

print(f"Matched {len(path_by_uid)}/{len(selected_uids)} selected files in the dataset listing.")

In [ ]:
DICOM_DIR = Path("data/siim_subset/dicom")
DICOM_DIR.mkdir(parents=True, exist_ok=True)

done = 0
for uid, remote_path in path_by_uid.items():
    out_file = DICOM_DIR / f"{uid}.dcm"
    if out_file.exists():
        done += 1
        continue
    for attempt in range(5):
        try:
            api.dataset_download_file(DATASET, remote_path, path=str(DICOM_DIR), force=True, quiet=True)
            downloaded = DICOM_DIR / Path(remote_path).name
            if downloaded != out_file and downloaded.exists():
                downloaded.rename(out_file)
            break
        except Exception as e:
            time.sleep(3 * (attempt + 1))
    done += 1
    if done % 50 == 0:
        print(f"{done}/{len(path_by_uid)}")

print("Downloaded:", len(list(DICOM_DIR.glob("*.dcm"))), "files")

## Build the dataset and dataloaders

In [ ]:
import sys
sys.path.insert(0, ".")

from torch.utils.data import DataLoader
from dicom_seg.dataset import build_manifest, train_val_split, SIIMPneumothoraxDataset

CSV_PATH = Path("data/siim_subset/stage_2_train.csv")

entries = build_manifest(CSV_PATH, DICOM_DIR)
train_entries, val_entries = train_val_split(entries, val_fraction=0.2, seed=42)
print(f"{len(entries)} images -> {len(train_entries)} train / {len(val_entries)} val")
print(f"Train positive: {sum(e.has_mask for e in train_entries)}, "
      f"Val positive: {sum(e.has_mask for e in val_entries)}")

train_ds = SIIMPneumothoraxDataset(train_entries, target_size=256)
val_ds = SIIMPneumothoraxDataset(val_entries, target_size=256)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

## Mount Google Drive (for checkpoints)

Checkpoints are saved periodically so a disconnected Colab runtime doesn't lose training progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = Path("/content/drive/MyDrive/medical_imaging_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)

## Train

Combined Dice+BCE loss (Milestone 5), Adam at lr=1e-4. Tracks train/val loss and val Dice/IoU every epoch to watch for overfitting.

In [ ]:
import torch
from torch.optim import Adam

from dicom_seg.unet import UNet
from dicom_seg.losses import DiceBCELoss
from dicom_seg.metrics import dice_coefficient, iou_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = UNet(in_channels=1, out_channels=1).to(device)
criterion = DiceBCELoss()
optimizer = Adam(model.parameters(), lr=1e-4)

EPOCHS = 30
history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    dice_scores, iou_scores = [], []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            loss = criterion(preds, masks)
            val_loss += loss.item() * images.size(0)
            for p, m in zip(preds, masks):
                dice_scores.append(dice_coefficient(p, m))
                iou_scores.append(iou_score(p, m))
    val_loss /= len(val_ds)
    val_dice = sum(dice_scores) / len(dice_scores)
    val_iou = sum(iou_scores) / len(iou_scores)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)
    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_dice={val_dice:.4f}  val_iou={val_iou:.4f}")

    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        ckpt_path = CHECKPOINT_DIR / f"unet_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), ckpt_path)
        print("  saved checkpoint:", ckpt_path)

## Loss / Dice curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Dice+BCE loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["val_dice"], label="val Dice")
axes[1].plot(history["val_iou"], label="val IoU")
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("epoch")
axes[1].legend()
fig.tight_layout()
fig.savefig("training_curves.png", dpi=150)
plt.show()

## Final evaluation vs. the naive baseline

Same validation split, same metrics as the local `scripts/evaluate_baseline.py` run recorded in `BENCHMARKS.md` (Otsu-threshold baseline: Dice 0.0190 overall / 0.0379 positive-only) -- so this is an apples-to-apples comparison, not just a training-run number.

In [ ]:
import cv2
import numpy as np

from dicom_seg.baseline import naive_threshold_baseline
from dicom_seg.preprocessing import dicom_to_image
from dicom_seg.rle import rle_decode_multi

model.eval()
unet_dice, unet_iou = [], []
unet_dice_pos, unet_iou_pos = [], []
baseline_dice_pos, baseline_iou_pos = [], []

with torch.no_grad():
    for entry in val_entries:
        image = dicom_to_image(entry.dicom_path, output_range="unit")
        target = rle_decode_multi(entry.rle_strings, height=image.shape[0], width=image.shape[1])

        image_256 = cv2.resize(image, (256, 256))
        image_t = torch.from_numpy(image_256).float().unsqueeze(0).unsqueeze(0).to(device)
        pred = model(image_t)[0, 0].cpu().numpy()
        pred_resized = cv2.resize(pred, (image.shape[1], image.shape[0]))

        d, i = dice_coefficient(pred_resized, target), iou_score(pred_resized, target)
        unet_dice.append(d)
        unet_iou.append(i)
        if entry.has_mask:
            unet_dice_pos.append(d)
            unet_iou_pos.append(i)
            baseline_pred = naive_threshold_baseline(image)
            baseline_dice_pos.append(dice_coefficient(baseline_pred, target))
            baseline_iou_pos.append(iou_score(baseline_pred, target))

print(f"U-Net      -- Dice (all): {np.mean(unet_dice):.4f}  IoU (all): {np.mean(unet_iou):.4f}")
print(f"U-Net      -- Dice (pos-only): {np.mean(unet_dice_pos):.4f}  IoU (pos-only): {np.mean(unet_iou_pos):.4f}")
print(f"Baseline   -- Dice (pos-only): {np.mean(baseline_dice_pos):.4f}  IoU (pos-only): {np.mean(baseline_iou_pos):.4f}")
print("\nCopy these numbers into BENCHMARKS.md's U-Net row.")

## Seg-Grad-CAM examples

Adapted Grad-CAM (Milestone 7): backpropagates from the summed predicted-mask logits through the U-Net bottleneck, producing a heatmap of which regions drove the prediction.

In [ ]:
from dicom_seg.gradcam import SegGradCAM

cam_tool = SegGradCAM(model, target_layer=model.bottleneck)

positive_val = [e for e in val_entries if e.has_mask][:4]
fig, axes = plt.subplots(3, len(positive_val), figsize=(4 * len(positive_val), 9))

for col, entry in enumerate(positive_val):
    image = dicom_to_image(entry.dicom_path, output_range="unit")
    target = rle_decode_multi(entry.rle_strings, height=image.shape[0], width=image.shape[1])
    image_256 = cv2.resize(image, (256, 256))
    image_t = torch.from_numpy(image_256).float().unsqueeze(0).unsqueeze(0).to(device)

    cam, probs = cam_tool(image_t)
    pred_mask = (probs[0, 0].cpu().numpy() > 0.5).astype(np.uint8)

    axes[0, col].imshow(image_256, cmap="gray")
    axes[0, col].set_title("input")
    axes[0, col].axis("off")

    axes[1, col].imshow(image_256, cmap="gray")
    axes[1, col].imshow(np.ma.masked_where(pred_mask == 0, pred_mask), cmap="autumn", alpha=0.5)
    axes[1, col].set_title("predicted mask")
    axes[1, col].axis("off")

    axes[2, col].imshow(image_256, cmap="gray")
    axes[2, col].imshow(cam, cmap="jet", alpha=0.4)
    axes[2, col].set_title("Seg-Grad-CAM")
    axes[2, col].axis("off")

fig.tight_layout()
fig.savefig("gradcam_examples.png", dpi=150)
plt.show()

## Bring results back to the local project

Download `training_curves.png` and `gradcam_examples.png` (left sidebar > Files, right-click > Download) into the local `outputs/` folder, and copy the printed Dice/IoU numbers into `BENCHMARKS.md`. The final checkpoint in `CHECKPOINT_DIR` (on Drive) is the trained model.